# BirdCLEF 2026: эксперименты по ячейкам

Этот notebook нужен для локального обучения и экспериментов в VS Code. Стабильный скрипт остается в `notebooks/bc2026_baseline.py`, а здесь можно запускать шаги по одному и смотреть, что происходит.

## 1. Импорты и путь к данным

Если у другого участника команды датасет лежит в другом месте, нужно поменять только `BC2026_DATA_DIR`.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "notebooks"))

os.environ.setdefault(
    "BC2026_DATA_DIR",
    "/Users/kseniadragun/Desktop/Predictive_Analytics/birds /birdclef-2026",
)

from bc2026_baseline import (
    CFG,
    build_training_table,
    find_data_dir,
    predict_submission,
    seed_everything,
    split_train_val,
    train_model,
)

print(f"project_root={PROJECT_ROOT}")
print(f"torch={torch.__version__}")

project_root=/Users/kseniadragun/Desktop/Predictive_Analytics/birdclef_project_files_v3/notebooks
torch=2.8.0


## 2. Конфиг

Для первого запуска оставляем `FAST_DEV = True`: notebook возьмет только 512 примеров и быстро проверит pipeline.

In [2]:
FAST_DEV = True

cfg = CFG()
if FAST_DEV:
    cfg.epochs = 1
    cfg.batch_size = 16

seed_everything(cfg.seed)
data_dir = find_data_dir()

print(f"data_dir={data_dir}")
print(f"device={cfg.device}")
print(f"epochs={cfg.epochs}")
print(f"batch_size={cfg.batch_size}")

data_dir=/Users/kseniadragun/Desktop/Predictive_Analytics/birds /birdclef-2026
device=mps
epochs=1
batch_size=16


## 3. Посмотрим основные таблицы

In [3]:
train_csv = pd.read_csv(data_dir / "train.csv")
taxonomy = pd.read_csv(data_dir / "taxonomy.csv")
sample = pd.read_csv(data_dir / "sample_submission.csv")

labels = [c for c in sample.columns if c != "row_id"]

print(f"train rows: {len(train_csv):,}")
print(f"taxonomy rows: {len(taxonomy):,}")
print(f"submission classes: {len(labels):,}")

taxonomy["class_name"].value_counts(dropna=False)

train rows: 35,549
taxonomy rows: 234
submission classes: 234


class_name
Aves        162
Amphibia     35
Insecta      28
Mammalia      8
Reptilia      1
Name: count, dtype: int64

In [4]:
train_csv.head()

,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat
3,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/818781.m...,1161364/iNat818781.ogg,iNat
4,1161364,[],[],-22.7426,-46.8985,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/556514.m...,1161364/iNat556514.ogg,iNat


## 4. Собираем обучающую таблицу

Здесь объединяются два источника:

- `train_audio`: короткие записи отдельных видов;
- `train_soundscapes_labels.csv`: размеченные 5-секундные куски настоящих soundscape-записей.

In [5]:
train_table = build_training_table(data_dir, labels)

if FAST_DEV:
    train_table = train_table.sample(
        n=min(len(train_table), cfg.max_train_rows_fast_dev),
        random_state=cfg.seed,
    ).reset_index(drop=True)

print(f"train_examples={len(train_table):,}")
train_table["source"].value_counts(dropna=False)

train_examples=512


source
train_audio         484
train_soundscape     28
Name: count, dtype: int64

In [6]:
train_table.head()

,path,start,end,labels,source,primary
0,/Users/kseniadragun/Desktop/Predictive_Analyti...,NaN,NaN,[gycwor1],train_audio,gycwor1
1,/Users/kseniadragun/Desktop/Predictive_Analyti...,NaN,NaN,[bkhpar],train_audio,bkhpar
2,/Users/kseniadragun/Desktop/Predictive_Analyti...,NaN,NaN,[phecuc1],train_audio,phecuc1
3,/Users/kseniadragun/Desktop/Predictive_Analyti...,NaN,NaN,[houspa],train_audio,houspa
4,/Users/kseniadragun/Desktop/Predictive_Analyti...,NaN,NaN,[chbmoc1],train_audio,chbmoc1


## 5. Train/validation split

Модель учится на `train_rows`, а на `val_rows` мы смотрим, уменьшается ли ошибка на данных, которые модель не видела во время обучения.

In [7]:
train_rows, val_rows = split_train_val(train_table, cfg.seed)
print(f"train={len(train_rows):,}")
print(f"val={len(val_rows):,}")

train=482
val=30


## 6. Обучение модели

В fast-режиме это короткая техническая проверка. Для более серьезного запуска можно поставить `FAST_DEV = False`, но на Mac это будет заметно дольше.

In [8]:
model = train_model(train_rows, val_rows, labels, cfg)

epoch=1 train_loss=0.1769 val_loss=0.0360


## 7. Сохраняем веса модели

Файлы в `outputs/` не коммитим в GitHub. Это локальные результаты экспериментов.

In [10]:
output_dir = PROJECT_ROOT / "outputs"
output_dir.mkdir(exist_ok=True)

model_path = output_dir / "baseline_fast_model.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "labels": labels,
        "cfg": cfg.__dict__,
    },
    model_path,
)

print(f"saved {model_path}")

saved /Users/kseniadragun/Desktop/Predictive_Analytics/birdclef_project_files_v3/notebooks/outputs/baseline_fast_model.pt


## 8. Submission

Локально настоящих hidden test audio нет, поэтому будет создана нейтральная заглушка. На Kaggle в этой же логике появятся настоящие predictions по `test_soundscapes`.

In [11]:
test_dir = data_dir / "test_soundscapes"
has_test_audio = test_dir.exists() and any(test_dir.glob("*.ogg"))

if has_test_audio:
    submission = predict_submission(model, data_dir, labels, cfg)
else:
    print("No test soundscapes found. Writing neutral sample-shaped submission.")
    submission = sample.copy()
    submission.loc[:, labels] = 0.01

submission_path = PROJECT_ROOT / "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"wrote {submission_path}")
submission.head()

No test soundscapes found. Writing neutral sample-shaped submission.
wrote /Users/kseniadragun/Desktop/Predictive_Analytics/birdclef_project_files_v3/notebooks/submission.csv


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,...,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01
1,BC2026_Test_0001_S05_20250227_010002_10,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,...,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01
2,BC2026_Test_0001_S05_20250227_010002_15,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,...,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01
